In [2]:
# KV-Cache Reuse Exploration
#
# **Setup:** Qwen3-0.6B and Qwen3-1.7B both have 28 layers, same KV shape [8, seq, 128].
# CKA shows high similarity for first 6 and last 10 layers.
#
# **Finding:** Keys have high cross-model cosine similarity (~0.95) but values have
# near-zero similarity (~0.06-0.10). Freezing both K+V produces garbage; freezing
# only keys works because attention = softmax(Q * K^T) * V, and V stays in the
# small model's representational space.
#
# **Approach:** Run the small model's prefill, but at frozen layers replace the
# computed keys with the large model's keys. Values are always computed by the
# small model. Non-frozen layers see the frozen keys through attention.
#
# **Goal:** Measure whether injecting large-model keys at selected layers improves
# the small model's HumanEval accuracy.

In [1]:
import re
import signal
import json
from pathlib import Path
from contextlib import contextmanager

import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.cache_utils import DynamicCache

/home/xz957/.conda/envs/sglang-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [2]:
MODEL_SMALL = "Qwen/Qwen3-0.6B"  # target model (runs generation)
MODEL_LARGE = "Qwen/Qwen3-1.7B"  # donor model (provides KV cache for selected layers)

# Layers to freeze from the large model during small model's prefill
# Both models have 28 layers (0-27), same KV shape [8, seq, 128]
REPLACE_LAYERS = list(range(0, 2))  # first 2 layers

MAX_NEW_TOKENS = 1024
NUM_SAMPLES = 1  # None = all 164
TIMEOUT = 5
DEVICE = "cuda:0"

print(f"Small model: {MODEL_SMALL}")
print(f"Large model: {MODEL_LARGE}")
print(f"Frozen layers: {REPLACE_LAYERS}")
print(f"Device: {DEVICE}")

Small model: Qwen/Qwen3-0.6B
Large model: Qwen/Qwen3-1.7B
Frozen layers: [0, 1]
Device: cuda:0


## Utilities

In [3]:
class TimeoutError(Exception):
    pass


@contextmanager
def time_limit(seconds: int):
    def handler(signum, frame):
        raise TimeoutError("Timed out!")
    signal.signal(signal.SIGALRM, handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)


def extract_code(text: str) -> str:
    """Extract python code from model output (```python blocks or raw text)."""
    match = re.search(r"```(?:python)?\s*\n(.*?)```", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()


def check_correctness(problem: dict, code: str, timeout: int = 5) -> dict:
    """Run extracted code against HumanEval tests."""
    task_id = problem["task_id"]
    full_code = code + "\n\n" + problem["test"] + f"\ncheck({problem['entry_point']})\n"
    try:
        with time_limit(timeout):
            exec(full_code, {})
        return {"task_id": task_id, "passed": True, "error": None}
    except TimeoutError:
        return {"task_id": task_id, "passed": False, "error": "Timed out"}
    except Exception as e:
        return {"task_id": task_id, "passed": False, "error": str(e)}


def format_prompt(problem_prompt: str) -> str:
    return (
        "Complete the following Python function. "
        "Return ONLY the complete function implementation in a python code block.\n\n"
        + problem_prompt
    )


def generate_from_kv(model, tokenizer, input_ids, past_key_values,
                     max_new_tokens=1024):
    """
    Generate using an externally-provided KV cache.

    Crop the cache to N-1 tokens, feed the Nth token as input_ids with
    explicit cache_position and an attention_mask covering all N positions
    (cached + current). This forces generate() to run a single-token
    "prefill" that fills the Nth KV slot, then the normal decode loop runs.
    """
    seq_len = past_key_values.get_seq_length()
    past_key_values.crop(seq_len - 1)
    last_token = input_ids[:, -1:]
    cache_position = torch.tensor([seq_len - 1], device=input_ids.device)
    # Attention mask must cover all positions: N-1 cached + 1 current
    attention_mask = torch.ones(1, seq_len, device=input_ids.device, dtype=torch.long)

    with torch.no_grad():
        out = model.generate(
            input_ids=last_token,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            cache_position=cache_position,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

## Load Models & Dataset

In [4]:
torch.cuda.empty_cache()

print("Loading tokenizers and models...")
tok_small = AutoTokenizer.from_pretrained(MODEL_SMALL, trust_remote_code=True)
model_small = AutoModelForCausalLM.from_pretrained(
    MODEL_SMALL, torch_dtype=torch.float16, device_map=DEVICE, trust_remote_code=True
).eval()

tok_large = AutoTokenizer.from_pretrained(MODEL_LARGE, trust_remote_code=True)
model_large = AutoModelForCausalLM.from_pretrained(
    MODEL_LARGE, torch_dtype=torch.float16, device_map=DEVICE, trust_remote_code=True
).eval()

print(f"Small model layers: {model_small.config.num_hidden_layers}")
print(f"Large model layers: {model_large.config.num_hidden_layers}")
print("Models loaded.")

Loading tokenizers and models...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|███████| 2/2 [00:00<00:00,  2.62it/s]


Small model layers: 28
Large model layers: 28
Models loaded.


In [5]:
dataset = load_dataset("trivia_qa", "rc", split="validation")
if NUM_SAMPLES is not None:
    dataset = dataset.select(range(min(NUM_SAMPLES, len(dataset))))
print(f"{len(dataset)} problems loaded")

# Context length target: we want seq_len > 2 * feat_dim (2048) to make SVD non-trivial
# Concatenate search contexts until we hit the target
CONTEXT_CHAR_LIMIT = 10000  # ~2500+ tokens after tokenization

def format_trivia_prompt(sample, char_limit=CONTEXT_CHAR_LIMIT):
    """Build a long-context QA prompt from TriviaQA sample."""
    contexts = sample["search_results"]["search_context"]
    context_text = ""
    for ctx in contexts:
        ctx_str = str(ctx)
        if len(context_text) + len(ctx_str) > char_limit:
            remaining = char_limit - len(context_text)
            if remaining > 100:
                context_text += ctx_str[:remaining]
            break
        context_text += ctx_str + "\n\n"
    return f"Context: {context_text}\n\nQuestion: {sample['question']}\nAnswer:"

1 problems loaded


## Sanity Check: Run One Problem

In [6]:
sample = dataset[0]
prompt = format_trivia_prompt(sample)

messages = [{"role": "user", "content": prompt}]
input_text = tok_small.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
)
inputs_small = tok_small(input_text, return_tensors="pt").to(DEVICE)
inputs_large = tok_large(input_text, return_tensors="pt").to(DEVICE)

print(f"Prompt length: {len(prompt)} chars, {inputs_small['input_ids'].shape[1]} tokens")
print(f"Question: {sample['question']}")
print(f"Expected answer: {sample['answer']['value']}")

# Prefill both models
with torch.no_grad():
    out_small = model_small(**inputs_small, use_cache=True)
    out_large = model_large(**inputs_large, use_cache=True)

kv_small = out_small.past_key_values
kv_large = out_large.past_key_values

print(f"\nSmall model KV: {len(kv_small)} layers, shape: {kv_small[0][0].shape}")
print(f"Large model KV: {len(kv_large)} layers, shape: {kv_large[0][0].shape}")
print(f"seq_len={kv_small[0][0].shape[2]}, feat_dim={kv_small[0][0].shape[1] * kv_small[0][0].shape[3]}")

# Cross-model KV compatibility check
print(f"\nCross-model KV compatibility (cosine similarity):")
print(f"{'Layer':>6} {'Keys cos':>10} {'Vals cos':>10}")
for li in range(0, 28, 3):
    ks, kl = kv_small[li][0].flatten().float(), kv_large[li][0].flatten().float()
    vs, vl = kv_small[li][1].flatten().float(), kv_large[li][1].flatten().float()
    cos_k = torch.nn.functional.cosine_similarity(ks.unsqueeze(0), kl.unsqueeze(0)).item()
    cos_v = torch.nn.functional.cosine_similarity(vs.unsqueeze(0), vl.unsqueeze(0)).item()
    print(f"{li:>6} {cos_k:>10.4f} {cos_v:>10.4f}")

Prompt length: 10066 chars, 2641 tokens
Question: Who was the man behind The Chipmunks?
Expected answer: David Seville

Small model KV: 28 layers, shape: torch.Size([1, 8, 2641, 128])
Large model KV: 28 layers, shape: torch.Size([1, 8, 2641, 128])
seq_len=2641, feat_dim=1024

Cross-model KV compatibility (cosine similarity):
 Layer   Keys cos   Vals cos
     0     0.9678     0.0932
     3     0.9501     0.0861
     6     0.8919     0.1248
     9     0.8756     0.0836
    12     0.8890     0.0588
    15     0.9271     0.0813
    18     0.7111     0.1228
    21     0.7852     0.1382
    24     0.6903     0.0797
    27     0.8108     0.0930


## SVD Transformation

Instead of reusing kv_large for the smaller model directly, we want to take advantage of low-rank projection and transform the kvcache to the low-rank space and then transform back.

Steps:
1. Compute kv_large and kv_small
2. Concatenate kv tensors layerwise by head dimension, so the shape will be [seq_len, 2 * num_head * head_dim]
3. Run SVD on them, and trim half rank, so the rank will be r = num_head * head_dim
4. Reconstruct a new kv_small using kv_large and the svd matrices assuming that A is shared, and B can be splited into B1 and B2 for M1 and M2
5. Feed in kv_small for decode

In [ ]:
# SVD Transformation: reconstruct only KEY cache from kv_large; values stay from kv_small

num_layers = len(kv_small)
batch, num_heads, seq_len, head_dim = kv_small[0][0].shape
feat_dim = num_heads * head_dim
rank = feat_dim  # half of concatenated dim (2 * feat_dim)

print(f"SVD transfer (keys only): {num_layers} layers, {num_heads} heads, seq_len={seq_len}, head_dim={head_dim}")
print(f"Feature dim per model: {feat_dim}, SVD rank: {rank}")
print(f"seq_len ({seq_len}) vs 2*feat_dim ({2*feat_dim}): {'overdetermined ✓' if seq_len > 2*feat_dim else 'underdetermined ✗ (need longer prompts)'}")

svd_kv_layers = []

for li in range(num_layers):
    # --- Keys: SVD reconstruction from concatenated small+large ---
    T_small_k = kv_small[li][0].cpu().float()
    T_large_k = kv_large[li][0].cpu().float()

    M_small_k = T_small_k[0].permute(1, 0, 2).reshape(seq_len, feat_dim)
    M_large_k = T_large_k[0].permute(1, 0, 2).reshape(seq_len, feat_dim)

    M_cat = torch.cat([M_small_k, M_large_k], dim=1)

    U, S, Vh = torch.svd_lowrank(A=M_cat, q=1024)
    Vh = torch.diag(S) @ Vh.T[:1024, :]
    r = min(rank, len(S))
    Vh_r = Vh[:, 1024:]
    del S, Vh, M_cat

    M_small_k_hat = U @ Vh_r

    mse = ((M_small_k - M_small_k_hat) ** 2).mean().item()
    nmse = mse / M_small_k.var().item() if M_small_k.var().item() > 0 else float('inf')
    cos = torch.nn.functional.cosine_similarity(
        M_small_k.flatten().unsqueeze(0), M_small_k_hat.flatten().unsqueeze(0)
    ).item()
    del M_small_k, M_large_k

    T_recon_k = M_small_k_hat.reshape(seq_len, num_heads, head_dim).permute(1, 0, 2).unsqueeze(0)
    T_recon_k = T_recon_k.to(device=DEVICE, dtype=kv_small[li][0].dtype)
    del M_small_k_hat

    if li % 4 == 0:
        print(f"  Layer {li:2d} Key: MSE={mse:.6f}, NMSE={nmse:.4f}, cos={cos:.4f}")

    # --- Values: use original small model values (no reconstruction) ---
    T_val = kv_small[li][1]

    svd_kv_layers.append((T_recon_k, T_val))

# Build DynamicCache from reconstructed KV
svd_cache = DynamicCache()
for li in range(num_layers):
    svd_cache.update(svd_kv_layers[li][0], svd_kv_layers[li][1], li)
del svd_kv_layers

print(f"\nSVD cache (keys reconstructed, values from small model): {svd_cache.get_seq_length()} tokens, {len(svd_cache)} layers")

# Generate with SVD-reconstructed cache
svd_text = generate_from_kv(
    model_small, tok_small, inputs_small["input_ids"],
    DynamicCache.from_legacy_cache(svd_cache.to_legacy_cache()),
    max_new_tokens=128,
)
del svd_cache
torch.cuda.empty_cache()

# Also generate baseline for comparison
with torch.no_grad():
    out_baseline = model_small.generate(
        **inputs_small, max_new_tokens=128, do_sample=False,
        pad_token_id=tok_small.eos_token_id,
    )
baseline_text = tok_small.decode(out_baseline[0][inputs_small["input_ids"].shape[1]:], skip_special_tokens=True)

print(f"\nQuestion: {sample['question']}")
print(f"Expected: {sample['answer']['value']}")
print(f"\n--- Baseline (small only) ---")
print(baseline_text[:300])
print(f"\n--- SVD Reconstructed (keys only) ---")
print(svd_text[:300])

## Full Evaluation

Run three configurations on all HumanEval problems:
1. **Small only** (0.6B baseline)
2. **Hybrid** (0.6B + 1.7B KV at selected layers)
3. **Large only** (1.7B baseline)

In [ ]:
def evaluate_model(model, tokenizer, dataset, label, max_new_tokens=1024,
                   timeout=5, device="cuda", kv_override_fn=None):
    """
    Evaluate a model on HumanEval.

    kv_override_fn: optional callable(inputs) -> DynamicCache
        If provided, uses generate_from_kv with the injected cache.
    """
    results = []
    passed = 0

    for i, problem in enumerate(dataset):
        prompt = format_prompt(problem["prompt"])
        messages = [{"role": "user", "content": prompt}]
        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        inputs = tokenizer(input_text, return_tensors="pt").to(device)

        with torch.no_grad():
            if kv_override_fn is not None:
                kv = kv_override_fn(inputs)
                generated_text = generate_from_kv(
                    model, tokenizer, inputs["input_ids"], kv,
                    max_new_tokens=max_new_tokens,
                )
            else:
                out = model.generate(
                    **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )
                generated_text = tokenizer.decode(
                    out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
                )

        code = extract_code(generated_text)
        result = check_correctness(problem, code, timeout=timeout)
        results.append(result)
        if result["passed"]:
            passed += 1

        if (i + 1) % 20 == 0 or i == len(dataset) - 1:
            print(f"  [{label}] {i+1}/{len(dataset)} — running pass@1: {passed/(i+1)*100:.1f}%")

    accuracy = passed / len(dataset) * 100
    print(f"  [{label}] Final pass@1: {accuracy:.2f}% ({passed}/{len(dataset)})")
    return results, accuracy

In [ ]:
# 1. Small model baseline
print("=== Evaluating small model (0.6B) ===")
results_small, acc_small = evaluate_model(
    model_small, tok_small, dataset, "0.6B",
    max_new_tokens=MAX_NEW_TOKENS, timeout=TIMEOUT, device=DEVICE,
)

=== Evaluating small model (0.6B) ===
  [0.6B] Final pass@1: 0.61% (1/164)


In [ ]:
# 2. Hybrid: small model with frozen keys from large model
def hybrid_kv_fn(inputs_batch):
    """
    1. Prefill large model to get donor keys
    2. Build FrozenKeysCache with donor keys at frozen layers
    3. Run small model's forward — frozen layers use donor keys,
       values always computed by small model
    """
    with torch.no_grad():
        out_l = model_large(
            input_ids=inputs_batch["input_ids"],
            attention_mask=inputs_batch.get("attention_mask", None),
            use_cache=True,
        )
        frozen_cache = build_frozen_keys_cache(out_l.past_key_values, REPLACE_LAYERS)
        out_s = model_small(**inputs_batch, use_cache=True, past_key_values=frozen_cache)
    return DynamicCache.from_legacy_cache(out_s.past_key_values.to_legacy_cache())

print(f"=== Evaluating hybrid (0.6B + frozen keys at layers {REPLACE_LAYERS} from 1.7B) ===")
results_hybrid, acc_hybrid = evaluate_model(
    model_small, tok_small, dataset, "hybrid",
    max_new_tokens=MAX_NEW_TOKENS, timeout=TIMEOUT, device=DEVICE,
    kv_override_fn=hybrid_kv_fn,
)

In [ ]:
# 3. Large model baseline
print("=== Evaluating large model (1.7B) ===")
results_large, acc_large = evaluate_model(
    model_large, tok_large, dataset, "1.7B",
    max_new_tokens=MAX_NEW_TOKENS, timeout=TIMEOUT, device=DEVICE,
)